# vLLM on a Kaggle T4

One-off check from SPEC section 4. The T4 is compute capability 7.5: no bf16, so fp16 only. Stop after 30 minutes of trying if it does not start cleanly, and record the outcome in `ENGINEERING_LOG.md`.

The pinned vLLM 0.10.1.1 requires Python 3.12 or older and the Kaggle kernel is newer, so vLLM gets its own Python 3.12 environment made with `uv`.

Settings: Internet on, accelerator GPU T4 x1.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv

In [ ]:
VLLM_ENV = "/kaggle/working/vllm-env"
!pip install -q uv
!uv venv --python 3.12 $VLLM_ENV
!uv pip install --python $VLLM_ENV/bin/python vllm==0.10.1.1
!$VLLM_ENV/bin/python --version
!$VLLM_ENV/bin/python -c "import vllm, torch; print(vllm.__version__, torch.__version__)"

In [ ]:
import subprocess

server = subprocess.Popen(
    f"{VLLM_ENV}/bin/vllm serve HuggingFaceTB/SmolVLM-256M-Instruct --dtype half"
    " --max-model-len 4096 --gpu-memory-utilization 0.8 --port 8000 > vllm.log 2>&1",
    shell=True,
)

In [ ]:
import time

import httpx

deadline = time.time() + 600
while time.time() < deadline:
    if server.poll() is not None:
        raise RuntimeError(f"vLLM exited with code {server.returncode}, see vllm.log")
    try:
        if httpx.get("http://localhost:8000/health", timeout=2).status_code == 200:
            break
    except httpx.TransportError:
        pass
    time.sleep(5)
else:
    raise TimeoutError("vLLM did not become healthy in 10 minutes, see vllm.log")
print("healthy")

In [ ]:
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg"
start = time.time()
response = httpx.post(
    "http://localhost:8000/v1/chat/completions",
    timeout=120,
    json={
        "model": "HuggingFaceTB/SmolVLM-256M-Instruct",
        "max_tokens": 64,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": image_url}},
                    {"type": "text", "text": "Describe this image in one sentence."},
                ],
            }
        ],
    },
)
print(f"{time.time() - start:.2f} s")
print(response.status_code, response.json())

In [ ]:
!tail -n 40 vllm.log